In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('/disk1/AI_Models/KcatPrediction/KEEN/data/combined_brenda_sabiork_geometric_mean.csv')

In [3]:
df.columns

Index(['substrate', 'organism', 'pH', 'temperature', 'type', 'sequence',
       'turnover_number', 'ec_number', 'uniprot', 'mutation',
       'isomeric_smiles', 'canonical_smiles'],
      dtype='object')

In [ ]:
print(f"Data length: {len(df)} rows")

pairs = df.groupby(['sequence', 'canonical_smiles']).size().reset_index(name='n')
print(f"Unique (sequence, substrate) pairs: {len(pairs)}")

rng = np.random.default_rng(42)
pair_idx = rng.permutation(len(pairs))

n_test  = int(len(pairs) * 0.1)
n_valid = int(len(pairs) * 0.1)

test_pairs  = set(zip(pairs.iloc[pair_idx[:n_test]]['sequence'],
                      pairs.iloc[pair_idx[:n_test]]['canonical_smiles']))
valid_pairs = set(zip(pairs.iloc[pair_idx[n_test:n_test+n_valid]]['sequence'],
                      pairs.iloc[pair_idx[n_test:n_test+n_valid]]['canonical_smiles']))
train_pairs = set(zip(pairs.iloc[pair_idx[n_test+n_valid:]]['sequence'],
                      pairs.iloc[pair_idx[n_test+n_valid:]]['canonical_smiles']))

pair_keys = list(zip(df['sequence'], df['canonical_smiles']))
split_labels = []
for p in pair_keys:
    if p in test_pairs:
        split_labels.append('test')
    elif p in valid_pairs:
        split_labels.append('valid')
    else:
        split_labels.append('train')
df['split'] = split_labels

leak = test_pairs & (train_pairs | valid_pairs)
print(f"\nTest ↔ Train/Valid pair leakage: {len(leak)} (must be 0)")

counts = df['split'].value_counts()
for s in ['train', 'valid', 'test']:
    print(f"  {s:6s}: {counts[s]:6d} ({counts[s]/len(df)*100:.1f}%)")

for split in ['train', 'valid', 'test']:
    idx_list = df[df['split'] == split].index.tolist()
    fname = f"data/{split}_seq_ids_new.txt"
    with open(fname, 'w') as f:
        f.write("\n".join(map(str, idx_list)))
    print(f"Saved {fname}: {len(idx_list)} indices")

총 데이터: 14711 rows
Unique (sequence, substrate) pairs: 12309

Test ↔ Train/Valid pair 누출: 0개 (0이어야 함)
  train :  11826 (80.4%)
  valid :   1410 (9.6%)
  test  :   1475 (10.0%)
Saved data/train_seq_ids_new.txt: 11826 indices
Saved data/valid_seq_ids_new.txt: 1410 indices
Saved data/test_seq_ids_new.txt: 1475 indices


In [ ]:
pairs = df.groupby(['sequence', 'canonical_smiles']).size().reset_index(name='n')
rng = np.random.default_rng(42)
pair_idx = rng.permutation(len(pairs))

n_test  = int(len(pairs) * 0.1)
n_valid = int(len(pairs) * 0.1)

test_pairs  = set(zip(pairs.iloc[pair_idx[:n_test]]['sequence'],
                      pairs.iloc[pair_idx[:n_test]]['canonical_smiles']))
valid_pairs = set(zip(pairs.iloc[pair_idx[n_test:n_test+n_valid]]['sequence'],
                      pairs.iloc[pair_idx[n_test:n_test+n_valid]]['canonical_smiles']))

pair_keys = list(zip(df['sequence'], df['canonical_smiles']))
df['_split'] = [
    'test'  if p in test_pairs  else
    'valid' if p in valid_pairs else
    'train'
    for p in pair_keys
]

PH_LO, PH_HI     = 6.0, 9.0
TEMP_LO, TEMP_HI = 22.0, 60.0

def score(pH, temp):
    ph_miss   = (pH   == -1) or pd.isna(pH)
    temp_miss = (temp == -1) or pd.isna(temp)

    pH_in   = (not ph_miss)   and (PH_LO   < pH   <= PH_HI)
    temp_in = (not temp_miss) and (TEMP_LO < temp <= TEMP_HI)

    if pH_in and temp_in:
        return 0, 0.0

    if pH_in or temp_in:
        return 1, 0.0

    if ph_miss:
        ph_dist = 9999.0
    elif pH <= PH_LO:
        ph_dist = PH_LO - pH
    elif pH > PH_HI:
        ph_dist = pH - PH_HI
    else:
        ph_dist = 0.0

    if temp_miss:
        temp_dist = 9999.0
    elif temp <= TEMP_LO:
        temp_dist = TEMP_LO - temp
    elif temp > TEMP_HI:
        temp_dist = temp - TEMP_HI
    else:
        temp_dist = 0.0

    return 2, ph_dist + temp_dist

scores = df.apply(lambda r: score(r['pH'], r['temperature']), axis=1, result_type='expand')
df['_priority'] = scores[0]
df['_distance'] = scores[1]

best_indices = (
    df.sort_values(['_priority', '_distance'])
      .groupby(['_split', 'sequence', 'substrate'])
      .apply(lambda g: g.index[0])         
      .values
)
best_set = set(best_indices)

df['seq-sub-split'] = df.apply(
    lambda r: r['_split'] if r.name in best_set else 'omit', axis=1
)
df = df.drop(columns=['_split', '_priority', '_distance'])

print(df['seq-sub-split'].value_counts())
print(f"\n총 rows: {len(df)}")
print(f"선택됨: {(df['seq-sub-split'] != 'omit').sum()}")
print(f"omit:   {(df['seq-sub-split'] == 'omit').sum()}")

selected = df[df['seq-sub-split'] != 'omit']
test_p  = set(zip(selected[selected['seq-sub-split']=='test']['sequence'],
                  selected[selected['seq-sub-split']=='test']['substrate']))
tv_p    = set(zip(selected[selected['seq-sub-split'].isin(['train','valid'])]['sequence'],
                  selected[selected['seq-sub-split'].isin(['train','valid'])]['substrate']))
print(f"\nTest ↔ Train/Valid pair leakage: {len(test_p & tv_p)} (must be 0)")


seq-sub-split
train    11062
test      1381
valid     1341
omit       927
Name: count, dtype: int64

총 rows: 14711
선택됨: 13784
omit:   927

Test ↔ Train/Valid pair 누출: 0개 (0이어야 함)


/tmp/ipykernel_1192736/1070145842.py:75: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.index[0])          # 정렬 후 첫 번째 = 최선
